# HW2 Playground

Fill in TODOs as you work through the assignment.
Implement the required sections in `model.py`, and use this notebook to orchestrate and run your solution.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from hw2_loader import HW2DataLoader
from model import GradientBoostingModel

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


In [ ]:
# TODO: Load both datasets
loader = HW2DataLoader()

# Heart disease dataset
heart_path = Path('../data/heart.csv')
X_heart, y_heart = loader.get_heart_disease_data(csv_path=heart_path)
print(X_heart.shape, y_heart.value_counts().to_dict())

df_heart = pd.concat([X_heart, y_heart], axis=1)
df_heart = df_heart.drop_duplicates()

X_heart = df_heart.drop(columns=y_heart.name)
y_heart = df_heart[y_heart.name]

print("After dropping:", X_heart.shape)

# Cancer genomics dataset
cancer_path = Path('../data/cancer_genomics.csv')
labels_path = Path('../data/labels_cancer_genomics.csv')
X_cancer, y_cancer = loader.get_cancer_genomics_data(
    csv_path=cancer_path, labels_path=labels_path
)
print(X_cancer.shape, y_cancer.value_counts().to_dict())
print(X_cancer.shape)
print(y_cancer.value_counts())
print(X_cancer.isna().sum().sum())
print(X_cancer.duplicated().sum())


In [ ]:
# TODO: Initialize your model (adjust params)
model = GradientBoostingModel(
    task='classification',
    max_depth=3,
    learning_rate=0.1,
    n_estimators=50,
    use_scaler=False,
)


In [ ]:
# TODO: Train/test split + fit (heart)
X_train, X_test, y_train, y_test = model.train_test_split(X_heart, y_heart)
model.fit(X_train, y_train)


In [ ]:
# TODO: Evaluate (heart)
metrics = model.evaluate(X_test, y_test)
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")
# print metrics

In [ ]:
# TODO: Cross-validation (heart)
# cv_results = model.cross_validate(...)
# print metrics

cv_results = model.cross_validate(X_heart, y_heart, cv=5)

print("\nCross-Validation Results:")
for metric, values in cv_results.items():
    print(f"{metric}: mean={values['mean']:.4f}, std={values['std']:.4f}")

In [ ]:
# TODO: Feature importance (heart)
feature_importance = model.get_feature_importance(plot=True, top_n=20)
print(feature_importance.head(10))


In [ ]:
# TODO: Hyperparameter tuning (heart)
param_grid = {
    'max_depth': [2, 4, 6, 8],
    'n_estimators': [20, 50, 100, 200],
    'learning_rate': [0.005, 0.01, 0.05, 0.1],
}
tuning_results = model.tune_hyperparameters(X_heart, y_heart, param_grid)
print(tuning_results['best_params'])
print(tuning_results['best_score'])

cv_df = pd.DataFrame(tuning_results["cv_results"])

fig, axes = plt.subplots(1, 4, figsize=(20,5), sharey=True)
for i, lr in enumerate(sorted(cv_df["param_learning_rate"].unique())):
    subset = cv_df[cv_df["param_learning_rate"] == lr]
    heatmap_data = subset.pivot(index="param_max_depth", columns="param_n_estimators", values="mean_test_score")
    sns.heatmap(heatmap_data, annot=True, fmt=".4f", cmap="viridis", ax=axes[i])
    axes[i].set_title(f"learning_rate={lr}")
plt.tight_layout()
plt.show()

In [ ]:
# TODO: Train/evaluate on cancer dataset (multi-class)

cancer_model = GradientBoostingModel(task="classification")
X_cancer_train, X_cancer_test, y_cancer_train, y_cancer_test = cancer_model.train_test_split(X_cancer, y_cancer)
cancer_model.fit(X_cancer_train, y_cancer_train)
cancer_metrics = cancer_model.evaluate(X_cancer_test, y_cancer_test)
print("Cancer Dataset Metrics:")
for k, v in cancer_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# hyperparemeter tuning on cancer dataset
cancer_param_grid = {
    'max_depth': [2, 4, 6],
    'n_estimators': [10, 25, 50],
    'learning_rate': [0.01, 0.05, 0.1],
}
cancer_tuning_results = cancer_model.tune_hyperparameters(X_cancer, y_cancer, cancer_param_grid)
print("\nCancer Hyperparameter Tuning Results:")
print("Best Parameters:", cancer_tuning_results['best_params'])
print("Best Score:", cancer_tuning_results['best_score'])

cv_df = pd.DataFrame(cancer_tuning_results["cv_results"])

param_col = "param_model__learning_rate" if "param_model__learning_rate" in cv_df.columns else "param_learning_rate"

fig, axes = plt.subplots(1, 3, figsize=(20,5), sharey=True)
for i, lr in enumerate(sorted(cv_df[param_col].unique())):
    subset = cv_df[cv_df[param_col] == lr]
    heatmap_data = subset.pivot(index="param_model__max_depth", columns="param_model__n_estimators", values="mean_test_score")
    sns.heatmap(heatmap_data, annot=True, fmt=".4f", cmap="viridis", ax=axes[i])
    axes[i].set_title(f"learning_rate={lr}")


In [ ]:
top_features_df = cancer_model.get_feature_importance(plot=True, top_n=20)
print(top_features_df.head(10))

top_k_features = top_features_df['feature'].head(20).tolist()
X_cancer_topk = X_cancer[top_k_features]


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
# these would normally go at the top but tbh didn't want to rerun everything above just to add these imports

K = 7
top_k_features = top_features_df['feature'].head(K).tolist()
X_cancer_topk = X_cancer[top_k_features]

# yay metrics
def evaluate_model(model, X, y):
    y_pred = model.predict(X)
    metrics = {
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, average="macro"),
        "recall": recall_score(y, y_pred, average="macro"),
        "f1": f1_score(y, y_pred, average="macro"),
    }
    if len(set(y)) > 2:
        y_proba = model.predict_proba(X)
        metrics["roc_auc"] = roc_auc_score(y, y_proba, multi_class="ovr", average="macro")
    else:
        y_proba = model.predict_proba(X)[:, 1]
        metrics["roc_auc"] = roc_auc_score(y, y_proba)
    return metrics

X_cancer_train, X_cancer_test, y_cancer_train, y_cancer_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)
X_cancer_topk_train = X_cancer_train[top_k_features]
X_cancer_topk_test = X_cancer_test[top_k_features]

#full features

scaler_full = StandardScaler()
scaler_full.fit(X_cancer_train)
X_cancer_train_scaled = scaler_full.transform(X_cancer_train)
X_cancer_test_scaled = scaler_full.transform(X_cancer_test)

hw1_full = LogisticRegression(max_iter=5000)
hw1_full.fit(X_cancer_train_scaled, y_cancer_train)
full_metrics = evaluate_model(hw1_full, X_cancer_test_scaled, y_cancer_test)

# top k features

scaler_topk = StandardScaler()
scaler_topk.fit(X_cancer_topk_train)
X_cancer_topk_train_scaled = scaler_topk.transform(X_cancer_topk_train)
X_cancer_topk_test_scaled = scaler_topk.transform(X_cancer_topk_test)

hw1_topk = LogisticRegression(max_iter=5000)
hw1_topk.fit(X_cancer_topk_train_scaled, y_cancer_train)
topk_metrics = evaluate_model(hw1_topk, X_cancer_topk_test_scaled, y_cancer_test)
comparison = pd.DataFrame(
    [full_metrics, topk_metrics],
    index=["Full Features", f"Top-{K} Features"]
)

print(comparison)